In [1]:
import os
import sys
import json
import torch
import importlib


In [2]:
PROJECT_ROOT = r"F:\ALL_CODE\ZIBA\bert\bert_sens"
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("MODEL_DIR:", MODEL_DIR)


PROJECT_ROOT: F:\ALL_CODE\ZIBA\bert\bert_sens
SRC_DIR: F:\ALL_CODE\ZIBA\bert\bert_sens\src
MODEL_DIR: F:\ALL_CODE\ZIBA\bert\bert_sens\models


In [3]:
import tokenizer as tokenizer_module
import modeling as modeling_module

importlib.reload(tokenizer_module)
importlib.reload(modeling_module)

Tokenizer = tokenizer_module.Tokenizer
BertConfig = modeling_module.BertConfig
BertForSequenceClassification = modeling_module.BertForSequenceClassification


In [4]:
print("Tokenizer module path:")
print(tokenizer_module.__file__)

print("\nDoes Tokenizer have load_vocab?")
print(hasattr(Tokenizer, "load_vocab"))


Tokenizer module path:
F:\ALL_CODE\ZIBA\bert\bert_sens\src\tokenizer.py

Does Tokenizer have load_vocab?
True


In [5]:
required_files = [
    "best_model.pt",
    "tokenizer_vocab.json",
    "model_config.json",
    "threshold.json"
]

for fname in required_files:
    fpath = os.path.join(MODEL_DIR, fname)
    print(fname, "=>", os.path.exists(fpath), "|", fpath)


best_model.pt => True | F:\ALL_CODE\ZIBA\bert\bert_sens\models\best_model.pt
tokenizer_vocab.json => True | F:\ALL_CODE\ZIBA\bert\bert_sens\models\tokenizer_vocab.json
model_config.json => True | F:\ALL_CODE\ZIBA\bert\bert_sens\models\model_config.json
threshold.json => True | F:\ALL_CODE\ZIBA\bert\bert_sens\models\threshold.json


In [6]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda


In [7]:
class SentimentPredictor:
    def __init__(self, model_dir, device=None):
        self.model_dir = model_dir
        self.device = device if device is not None else torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        # 1) load model config
        config_path = os.path.join(model_dir, "model_config.json")
        with open(config_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)

        # 2) load threshold
        threshold_path = os.path.join(model_dir, "threshold.json")
        with open(threshold_path, "r", encoding="utf-8") as f:
            threshold_data = json.load(f)
        self.threshold = float(threshold_data["neutral_threshold"])

        # 3) load tokenizer vocab
        tokenizer_vocab_path = os.path.join(model_dir, "tokenizer_vocab.json")
        self.tokenizer = Tokenizer(vocab_size=cfg["vocab_size"])
        self.tokenizer.load_vocab(tokenizer_vocab_path)

        # 4) build model
        config = BertConfig(**cfg)
        self.model = BertForSequenceClassification(config).to(self.device)

        # 5) load weights
        model_path = os.path.join(model_dir, "best_model.pt")
        state_dict = torch.load(model_path, map_location=self.device)
        self.model.load_state_dict(state_dict)

        self.model.eval()

        self.id2label = {
            0: "منفی",
            1: "مثبت",
            2: "خنثی"
        }

        print("Model loaded successfully.")
        print("Device:", self.device)
        print("Neutral threshold:", self.threshold)

    def predict(self, text, max_length=128):
        enc = self.tokenizer.encode_plus(
            text=text,
            add_special_tokens=True,
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        input_ids = enc["input_ids"].unsqueeze(0).to(self.device)
        attention_mask = enc["attention_mask"].unsqueeze(0).to(self.device)

        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask)

            # چون در کد train شما این‌طور استفاده شده:
            # logits, _ = model(...)
            # پس اینجا هم همان الگو را نگه می‌داریم.
            logits, _ = outputs

            probs = torch.softmax(logits, dim=1).squeeze(0)

        # threshold gating برای کلاس خنثی
        if probs[2].item() >= self.threshold:
            pred = 2
        else:
            pred = torch.argmax(probs[:2]).item()

        return {
            "text": text,
            "label_id": int(pred),
            "label": self.id2label[pred],
            "confidence": float(probs[pred].item()),
            "probabilities": {
                "منفی": float(probs[0].item()),
                "مثبت": float(probs[1].item()),
                "خنثی": float(probs[2].item())
            }
        }

    def predict_batch(self, texts, max_length=128):
        results = []
        for text in texts:
            results.append(self.predict(text, max_length=max_length))
        return results


In [8]:
predictor = SentimentPredictor(model_dir=MODEL_DIR, device=DEVICE)


Tokenizer vocab loaded from: F:\ALL_CODE\ZIBA\bert\bert_sens\models\tokenizer_vocab.json
Loaded vocab size: 10000
Model loaded successfully.
Device: cuda
Neutral threshold: 0.6000000000000001


In [9]:
text = "واقعا از این محصول راضی بودم، خیلی عالی کار می‌کنه!"
result = predictor.predict(text)

result


{'text': 'واقعا از این محصول راضی بودم، خیلی عالی کار می\u200cکنه!',
 'label_id': 1,
 'label': 'مثبت',
 'confidence': 0.9611775875091553,
 'probabilities': {'منفی': 0.025918588042259216,
  'مثبت': 0.9611775875091553,
  'خنثی': 0.012903823517262936}}

In [10]:
texts = [
    "واقعا از این محصول راضی بودم، خیلی عالی کار می‌کنه!",
    "اصلاً خوب نبود، پولم هدر رفت.",
    "محصول به دستم رسید، معمولی بود.",
    "کیفیتش بد نیست ولی چیز خاصی هم ندارد.",
    "فوق‌العاده بود و دوباره هم می‌خرم."
]

results = predictor.predict_batch(texts)

for i, res in enumerate(results, 1):
    print(f"--- Sample {i} ---")
    print("Text:", res["text"])
    print("Label:", res["label"])
    print("Confidence:", round(res["confidence"], 4))
    print("Probabilities:", res["probabilities"])
    print()


--- Sample 1 ---
Text: واقعا از این محصول راضی بودم، خیلی عالی کار می‌کنه!
Label: مثبت
Confidence: 0.9612
Probabilities: {'منفی': 0.025918588042259216, 'مثبت': 0.9611775875091553, 'خنثی': 0.012903823517262936}

--- Sample 2 ---
Text: اصلاً خوب نبود، پولم هدر رفت.
Label: منفی
Confidence: 0.9286
Probabilities: {'منفی': 0.9285928606987, 'مثبت': 0.06672975420951843, 'خنثی': 0.0046774097718298435}

--- Sample 3 ---
Text: محصول به دستم رسید، معمولی بود.
Label: مثبت
Confidence: 0.3535
Probabilities: {'منفی': 0.09846960008144379, 'مثبت': 0.3534712791442871, 'خنثی': 0.5480591654777527}

--- Sample 4 ---
Text: کیفیتش بد نیست ولی چیز خاصی هم ندارد.
Label: خنثی
Confidence: 0.7295
Probabilities: {'منفی': 0.24190391600131989, 'مثبت': 0.02854796312749386, 'خنثی': 0.7295480966567993}

--- Sample 5 ---
Text: فوق‌العاده بود و دوباره هم می‌خرم.
Label: مثبت
Confidence: 0.9743
Probabilities: {'منفی': 0.019992539659142494, 'مثبت': 0.9742648005485535, 'خنثی': 0.005742641631513834}

